### 1. Load Benchmark Problems

In [1]:
sampling_for_lite = False # True for MineCEraft Lite
sampling_rand_seed = 42

In [2]:
# --- Load prompts from the mapping (single source of truth) ---
from pathlib import Path
import json
import random

# Load JSON files from benchmarks folder only (not archive/; archive = excluded from evaluation)
# Schema: prompts = [[turn1, turn2, ...], ...], checks = [[eval_turn1], [eval_turn2], ...]
BENCHMARKS_DIR = Path.cwd() / "benchmarks"
json_files = sorted(BENCHMARKS_DIR.glob("*.json"))
mapping = []
for json_file in json_files:
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

# Build runs: each run = (prompt_sequence, checks_per_turn, _comment)
runs = []
if sampling_for_lite:
    rng = random.Random(sampling_rand_seed)
    for json_file in json_files:
        file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
        file_runs = []
        for item in file_mapping:
            for prompt_sequence in item["prompts"]:
                file_runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))
        if file_runs:
            runs.append(rng.choice(file_runs))
else:
    for item in mapping:
        for prompt_sequence in item["prompts"]:
            runs.append((prompt_sequence, item["checks"], item.get("_comment", "")))

print(f"[PY] Benchmarks dir: {BENCHMARKS_DIR.resolve()} (cwd: {Path.cwd().resolve()})")
if len(runs) == 0:
    print("[PY] ⚠️ No runs. Put at least one .json in benchmarks/ (not in archive/), or run notebook from the folder that contains benchmarks/.")
print("Total problem #:", len(runs))
for i, (prompts, _, _) in enumerate(runs):
    print(i, prompts)

[PY] Benchmarks dir: D:\git\mineCEraft\mineCEraft\benchmarks (cwd: D:\git\mineCEraft\mineCEraft)
Total problem #: 64
0 ['Lay the foundation for a 7x7 block rectangular building. Use stone blocks and make it one block deep.']
1 ['Install the foundation for a 7x7 block rectangular building. Use stone blocks and make it one block deep.']
2 ['Lay the foundation for a 7x7 block rectangular building. Use stone blocks and make it two blocks deep.']
3 ['Install the foundation for a 7x7 block rectangular building. Use stone blocks and make it two blocks deep.']
4 ['Lay the foundation for a 7x8 block rectangular building. Use stone blocks and make it one block deep.']
5 ['Install the foundation for a 7x8 block rectangular building. Use stone blocks and make it one block deep.']
6 ['Lay the foundation for a 7x8 block rectangular building. Use stone blocks and make it two blocks deep.']
7 ['Install the foundation for a 7x8 block rectangular building. Use stone blocks and make it two blocks deep.']

### 2. Build phase (construction)

This cell runs the construction (builder agent) only and writes an `eval_raw_*.json` file that contains, for each prompt, its checks and cumulative coordinates. Run the "Load Benchmark Problems" cell above first so that `runs` is defined.

In [3]:
from pathlib import Path
from build_eval_raw import run_build_and_save_eval_raw

# Build phase:
# - uses `runs` constructed in the "Load Benchmark Problems" cell
# - talks to the builder agent
# - writes a single eval_raw_{model_safe}_{ts}.json file under eval_results/

try:
    runs  # type: ignore[name-defined]
except NameError as exc:
    raise RuntimeError("`runs` is not defined. Run the 'Load Benchmark Problems' cell above first.") from exc

eval_raw_path = run_build_and_save_eval_raw(runs)
# For downstream evaluation we treat eval_raw as a list of files.
eval_raw_files = [eval_raw_path]

print(f"[PY] eval_raw file for evaluation: {eval_raw_path}")

[PY] Using builder model: gemini-3-pro-preview (safe='gemini-3-pro-preview')
[PY] Total turns to send: 64
[PY] Intermediate eval_raw file: eval_results\eval_raw_gemini-3-pro-preview_20260305_100920.json
[PY] Builder agent started (PID=5388)
ℹ️ Using agent name: builder
🧹 Cleared all files under D:\git\mineCEraft\bots\builder\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=BhamCnZ7kVew_kWIAAAD)

➡️ Sending to builder (run 1, turn 1/1): "Lay the foundation for a 7x7 block rectangular building. Use stone blocks and make it one block deep."
⏳ Waiting for completion keyword (timeout 10 min)...
📨 [builder]  !newAction("Build a 7x7 rectangular foundation filled with stone blocks starting at my current position (x=-5, z=-7) at y=-1 (replacing the floor) or y=0.")
📨 [builder] I've successfully built the 7x7 stone foundation. 
✅ Completion detected for "Lay the foundation for a 7x7 block rectangular building. Use stone blocks and make it one block deep.".
::ACTION_MAX_J

### 3. Evaluation phase (eval_raw → log/csv)

This cell reads one or more `eval_raw_*.json` files and produces `eval_{model_safe}_{ts}.log` and `eval_{model_safe}_{ts}.csv`. It also populates `coords_by_problem` for the visualization cell.

In [ ]:
from eval_from_raw import evaluate_from_raw

# Default: evaluate the eval_raw file produced in the build phase above.
coords_by_problem, log_path, csv_path = evaluate_from_raw(eval_raw_files)

# Optional: evaluate a custom list of eval_raw files instead of the default one.
# Example:
# manual_eval_raw_files = [
#     "eval_results\eval_raw_gemini-3-pro-preview_20260305_100920.json",
# ]
# coords_by_problem, log_path, csv_path = evaluate_from_raw(manual_eval_raw_files)

print(f"[PY] Log path: {log_path}")
print(f"[PY] CSV path: {csv_path}")

[PY] Reading eval_raw from eval_results\eval_raw_gemini-3-pro-preview_20260305_100920.json

[PY] === Evaluation Result ===
[PY] Run #1, Turn #1/1: Lay the foundation for a 7x7 block rectangular building. Use stone blocks and make it one block deep.
[PY] Score: 3.571428571428571 / 4 (coords=49)
[PY] Category scores:
  - efficiency: 0.5714285714285714 / 1
  - material: 1.0 / 1
  - physical_plausibility: 1.0 / 1
  - size: 1.0 / 1
  · PASS | eval_code.material.is_all_material_equal_to({'expected_material': 'stone'})
  · PASS | eval_code.size.is_equal({'xz': [7, 7], 'y': 1})
  · PASS | eval_code.physical_plausibility.is_ground_connected({})
  · 0.57 | eval_code.efficiency.l1_dist({})

[PY] === Evaluation Result ===
[PY] Run #2, Turn #1/1: Install the foundation for a 7x7 block rectangular building. Use stone blocks and make it one block deep.
[PY] Score: 3.571428571428571 / 4 (coords=49)
[PY] Category scores:
  - efficiency: 0.5714285714285714 / 1
  - material: 1.0 / 1
  - physical_plausibi